# Day 18: Inheritance, backtracking, Adam and LayerNorm

*Week 4: ML from scratch*

**Time budget:** about 60–90 minutes. **A** is Python fluency (~15 min), **B** is an interview-style algorithm (~25 min), **C** is NumPy / PyTorch / ML (~30 min).

**For each problem:**
1. Plan on paper or in comments before you type any code.
2. Work on it for 20 minutes before asking for a hint.
3. Run the test cell under it. Don't edit the tests.
4. Answer the **Say out loud** prompts, out loud. Interviews grade the explanation as much as the code.

In [ ]:
import numpy as np

## A. An optimizer class hierarchy

`Python`

Use **inheritance**, the way `torch.optim` is designed. `params` and `grads` are dicts from name to NumPy array.

1. `Optimizer(params, lr)`: the base class. Store both. `step(grads)` raises `NotImplementedError`.
2. `SGD(Optimizer)`: `step` does `p -= lr * g` for each parameter, **in place**. Don't rebind the dict entry to a new array, because other code may hold a reference to the old one.
3. `Momentum(Optimizer)`: add a `beta=0.9` argument. Keep a velocity per parameter, starting at zero: `v = beta * v + g`, then `p -= lr * v`. That's PyTorch's convention.

**Look up:** `super().__init__(...)`, `p -= ...` vs. `p = p - ...` on NumPy arrays (one is in place and one isn't), `np.zeros_like`.

In [ ]:
class Optimizer:
    # your code here
    pass


class SGD(Optimizer):
    # your code here
    pass


class Momentum(Optimizer):
    # your code here
    pass

In [ ]:
# Tests: run after your solution. Don't edit.
import torch
try:
    Optimizer({"w": np.zeros(2)}, lr=0.1).step({"w": np.zeros(2)})
    raise AssertionError("base class step should raise NotImplementedError")
except NotImplementedError:
    pass
params = {"w": np.array([1.0, 2.0])}
w_ref = params["w"]
SGD(params, lr=0.1).step({"w": np.array([1.0, 1.0])})
assert params["w"] is w_ref, "update in place"
assert np.allclose(w_ref, [0.9, 1.9])
assert issubclass(SGD, Optimizer) and issubclass(Momentum, Optimizer)
rng = np.random.default_rng(0)
w0 = rng.normal(size=4)
grads = [rng.normal(size=4) for _ in range(5)]
p = {"w": w0.copy()}
opt = Momentum(p, lr=0.1, beta=0.9)
tw = torch.tensor(w0.copy(), requires_grad=True)
topt = torch.optim.SGD([tw], lr=0.1, momentum=0.9)
for g in grads:
    opt.step({"w": g})
    tw.grad = torch.tensor(g); topt.step()
assert np.allclose(p["w"], tw.detach().numpy()), "momentum doesn't match torch.optim.SGD(momentum=0.9)"
print("✅ A passed")

## B. Backtracking: subsets and parentheses

`Algorithms`

1. `subsets(nums)`: all subsets of a list of distinct numbers, each as a list (any order).
2. `generate_parentheses(n)`: all valid strings of `n` pairs of parentheses.

The backtracking template: a recursive helper takes the current partial answer. At each step it tries every allowed choice, recurses, then **undoes** the choice.

For (2), when are you allowed to add `(`? When are you allowed to add `)`?

**Say out loud:** how many subsets are there? How many results for `generate_parentheses(n)`? (Look up the Catalan numbers.)

In [ ]:
def subsets(nums):
    # your code here
    pass


def generate_parentheses(n):
    # your code here
    pass

In [ ]:
# Tests: run after your solution. Don't edit.
canon = lambda ss: sorted(sorted(s) for s in ss)
assert canon(subsets([1, 2, 3])) == canon([[], [1], [2], [3], [1, 2], [1, 3], [2, 3], [1, 2, 3]])
assert subsets([]) == [[]]
assert len(subsets(list(range(10)))) == 1024
assert sorted(generate_parentheses(3)) == sorted(["((()))", "(()())", "(())()", "()(())", "()()()"])
assert generate_parentheses(1) == ["()"]
assert len(generate_parentheses(6)) == 132
print("✅ B passed")

## C. Adam and LayerNorm from scratch

`ML from scratch`

1. `class Adam` with `__init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8)` and `step(grads)`, updating `params` in place (same interface as Problem A, and you may subclass your `Optimizer`). Each step:
   - m = β₁m + (1−β₁)g, and v = β₂v + (1−β₂)g²
   - bias-correct: m̂ = m / (1−β₁ᵗ), v̂ = v / (1−β₂ᵗ), where t counts steps from 1
   - p −= lr · m̂ / (√v̂ + eps)
2. `layer_norm(x, gamma, beta, eps=1e-5)`: normalize over the **last** axis (mean 0, variance 1, using the *biased* variance), then scale by `gamma` and shift by `beta`.

**Say out loud:** why the bias correction? (What are m and v right after step 1?) And how does **batch** norm differ from layer norm: which axis does each normalize over, and why does batch norm behave differently in train and eval mode?

In [ ]:
class Adam:
    # your code here
    pass


def layer_norm(x, gamma, beta, eps=1e-5):
    # your code here
    pass

In [ ]:
# Tests: run after your solution. Don't edit.
import torch, torch.nn.functional as F
rng = np.random.default_rng(0)
w0 = rng.normal(size=(3, 2))
grads = [rng.normal(size=(3, 2)) for _ in range(6)]
p = {"w": w0.copy()}
opt = Adam(p, lr=0.01)
tw = torch.tensor(w0.copy(), requires_grad=True)
topt = torch.optim.Adam([tw], lr=0.01)
for g in grads:
    opt.step({"w": g})
    tw.grad = torch.tensor(g); topt.step()
assert np.allclose(p["w"], tw.detach().numpy(), atol=1e-7), "Adam doesn't match torch.optim.Adam"
x = rng.normal(size=(4, 5, 8)) * 3 + 1
gamma, beta = rng.normal(size=8), rng.normal(size=8)
expected = F.layer_norm(torch.tensor(x), (8,), torch.tensor(gamma), torch.tensor(beta), eps=1e-5).numpy()
assert np.allclose(layer_norm(x, gamma, beta), expected, atol=1e-6)
print("✅ C passed")

## Wrap-up (5 minutes)

- For each problem: what's its time and space complexity? Which edge cases did you handle?
- Anything that took more than 20 minutes, or needed a hint: add it to the **Redo list** in `README.md` and solve it again from scratch in 3 days, without looking at today's code.
- Tick off today in `README.md`.